<a href="https://colab.research.google.com/github/fidlarsyn/grokking-Deep-Learning-Chapter-7-to-10/blob/main/08_Learning_Signal_and_Ignoring_Noise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Three-Layer Network on MNIST**

Menguji jaringan tiga layer (*input*, *hidden layer* dengan aktivasi ReLU, dan *output layer*) pada dataset klasifikasi digit MNIST. Jaringan yang lebih dalam memiliki kapasitas ekspresi (*expressive power*) yang sangat besar, sehingga mampu mempelajari korelasi perantara (*intermediate correlation*) untuk menyelesaikan tugas klasifikasi yang kompleks. Namun, kapasitas ekspresi yang besar ini juga membuat jaringan sangat rentan terhadap *overfitting*.

# **Memorization vs. Generalization**

Jaringan saraf melatih bobotnya agar mampu mengonversi masukan spesifik menjadi prediksi spesifik. Jika jaringan dilatih pada 1.000 gambar dan berhasil memprediksinya 100% sempurna, hal itu belum menjamin model akan bekerja baik pada gambar baru. *Memorization* terjadi ketika jaringan hanya menghafal data latih, sedangkan *generalization* adalah kemampuan jaringan untuk memberikan prediksi yang akurat pada data baru (*testing set*) yang belum pernah dilihat sebelumnya.

# **Overfitting in Neural Networks &amp; Where Overfitting Comes From**

*Overfitting* didefinisikan sebagai kondisi di mana jaringan saraf mempelajari *noise* (detail acak/irrelevant) pada dataset latih, bukan hanya sinyal utama (*true signal*). Penulis mengilustrasikan *overfitting* seperti membuat cetakan garpu tiga gigi dari tanah liat; jika cetakan dibuat terlalu detail sesuai satu garpu spesifik, cetakan tersebut akan menolak garpu empat gigi meskipun keduanya sama-sama garpu. Pada jaringan saraf besar dengan jutaan parameter, jaringan cenderung menyesuaikan bentuk bobotnya mengikuti *noise* detail gambar latih.

# **The Simplest Regularization: Early Stopping**

*Regularization* adalah sekumpulan metode yang dirancang untuk membantu model melakukan generalisasi serta mengabaikan *noise*. Bentuk *regularization* paling sederhana dan murah adalah **early stopping**, yaitu menghentikan proses pelatihan saat performa pada dataset validasi (*validation set*) mulai menurun, sebelum jaringan sempat mempelajari *noise* tingkat tinggi.

# **Industry Standard Regularization: Dropout**

**Dropout** adalah teknik *regularization* standar industri di mana nodus-nodus pada *hidden layer* dimatikan secara acak (nilainya diset ke 0) selama proses pelatihan berlangsung. Hal ini memaksa jaringan untuk berlatih secara eksklusif menggunakan sub-bagian acak (*random subsections*) dari arsitektur jaringan.

# **Why Dropout Works: Ensembling Works**

Dropout bekerja karena dua alasan utama:
1. **Simulasi Jaringan Kecil**: Jaringan kecil tidak memiliki kapasitas parameter berlebih untuk menghafal *noise*, sehingga hanya mampu menangkap fitur-fitur utama. Dropout membuat jaringan besar berperilaku seperti ansambel jaringan-jaringan kecil.
2. **Ensembling &amp; Noise Cancellation**: Jaringan saraf yang diinisialisasi secara acak cenderung mempelajari *noise* yang berbeda-beda. Ketika hasil dari banyak sub-jaringan acak dirata-ratakan, *noise* individual saling menghilangkan (*cancel out*) dan hanya sinyal utama (*signal*) yang tersisa.

# **Dropout in Code &amp; Evaluation on MNIST**

Implementasi dropout dilakukan dengan mengalikan matriks aktivasi `layer_1` dengan `dropout_mask` (distribusi Bernoulli 50% yang berisi angka 0 dan 1). Karena separuh nodus dimatikan, nilai aktivasi yang tersisa dikalikan dengan 2 (`layer_1 *= dropout_mask * 2`) untuk menjaga skala bobot penumlahan (*volume*) tetap seimbang antara tahap *training* dan *testing*. Pada MNIST, tanpa dropout akurasi pengujian turun dari 81.14% menjadi 70.73% (*overfitting* parah), sedangkan dengan dropout akurasi pengujian mencapai \~82.36% dan tetap stabil (\~81.81%).

In [ ]:
import numpy as np
import sys

np.random.seed(1)

def relu(x):
    return (x >= 0) * x

def relu2deriv(output):
    return output >= 0

alpha, iterations, hidden_size = (0.005, 300, 100)
pixels_per_image, num_labels = (784, 10)

weights_0_1 = 0.2 * np.random.random((pixels_per_image, hidden_size)) - 0.1
weights_1_2 = 0.2 * np.random.random((hidden_size, num_labels)) - 0.1

for j in range(iterations):
    error, correct_cnt = (0.0, 0)
    for i in range(len(images)):
        layer_0 = images[i:i+1]
        layer_1 = relu(np.dot(layer_0, weights_0_1))

        # Penerapan Dropout Mask (50% Bernoulli)
        dropout_mask = np.random.randint(2, size=layer_1.shape)
        layer_1 *= dropout_mask * 2

        layer_2 = np.dot(layer_1, weights_1_2)

        error += np.sum((labels[i:i+1] - layer_2) ** 2)
        correct_cnt += int(np.argmax(layer_2) == np.argmax(labels[i:i+1]))

        layer_2_delta = (labels[i:i+1] - layer_2)
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)
        layer_1_delta *= dropout_mask

        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

    if (j % 10 == 0):
        test_error = 0.0
        test_correct_cnt = 0
        for i in range(len(test_images)):
            layer_0 = test_images[i:i+1]
            layer_1 = relu(np.dot(layer_0, weights_0_1))
            layer_2 = np.dot(layer_1, weights_1_2)
            test_error += np.sum((test_labels[i:i+1] - layer_2) ** 2)
            test_correct_cnt += int(np.argmax(layer_2) == np.argmax(test_labels[i:i+1]))

        sys.stdout.write("\\n" + \\
            "I:" + str(j) + \\
            " Test-Err:" + str(test_error / float(len(test_images)))[0:5] + \\
            " Test-Acc:" + str(test_correct_cnt / float(len(test_images))) + \\
            " Train-Err:" + str(error / float(len(images)))[0:5] + \\
            " Train-Acc:" + str(correct_cnt / float(len(images))))

# **Batch Gradient Descent (Mini-Batching)**

Dibandingkan memperbarui bobot pada setiap 1 sampel data (*Stochastic Gradient Descent / SGD*), *mini-batch gradient descent* memperbarui bobot setelah menghitung rata-rata pembaruan dari sekelompok sampel data (misalnya `batch_size = 100`). Mengambil rata-rata gradien dalam satu *batch* mengurangi *noise* pembaruan individual, menghasilkan kurva pelatihan yang lebih mulus, dan memungkinkan penggunaan nilai *learning rate* (`alpha`) yang lebih besar.

In [ ]:
import numpy as np
import sys

np.random.seed(1)

def relu(x):
    return (x >= 0) * x

def relu2deriv(output):
    return output >= 0

batch_size = 100
alpha, iterations = (0.001, 300)
pixels_per_image, num_labels, hidden_size = (784, 10, 100)

weights_0_1 = 0.2 * np.random.random((pixels_per_image, hidden_size)) - 0.1
weights_1_2 = 0.2 * np.random.random((hidden_size, num_labels)) - 0.1

for j in range(iterations):
    error, correct_cnt = (0.0, 0)
    for i in range(int(len(images) / batch_size)):
        batch_start, batch_end = ((i * batch_size), ((i + 1) * batch_size))

        layer_0 = images[batch_start:batch_end]
        layer_1 = relu(np.dot(layer_0, weights_0_1))

        dropout_mask = np.random.randint(2, size=layer_1.shape)
        layer_1 *= dropout_mask * 2

        layer_2 = np.dot(layer_1, weights_1_2)

        error += np.sum((labels[batch_start:batch_end] - layer_2) ** 2)
        for k in range(batch_size):
            correct_cnt += int(np.argmax(layer_2[k:k+1]) == \\
                np.argmax(labels[batch_start+k:batch_start+k+1]))

        layer_2_delta = (labels[batch_start:batch_end] - layer_2) / batch_size
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)
        layer_1_delta *= dropout_mask

        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

    if (j % 10 == 0):
        test_error = 0.0
        test_correct_cnt = 0
        for i in range(len(test_images)):
            layer_0 = test_images[i:i+1]
            layer_1 = relu(np.dot(layer_0, weights_0_1))
            layer_2 = np.dot(layer_1, weights_1_2)
            test_error += np.sum((test_labels[i:i+1] - layer_2) ** 2)
            test_correct_cnt += int(np.argmax(layer_2) == np.argmax(test_labels[i:i+1]))

        sys.stdout.write("\\n" + \\
            "I:" + str(j) + \\
            " Test-Err:" + str(test_error / float(len(test_images)))[0:5] + \\
            " Test-Acc:" + str(test_correct_cnt / float(len(test_images))) + \\
            " Train-Err:" + str(error / float(len(images)))[0:5] + \\
            " Train-Acc:" + str(correct_cnt / float(len(images))))